# Portfolio / Brokers — Dev Playground

Interactive notebook for exercising the `/portfolio/*` API surface end-to-end. Two paths are supported:

1. **In-process** — call the FastAPI app via `TestClient` (no server needed). Fastest iteration; good for parser debugging.
2. **Live HTTP** — point at a running `pdm run dev` server. Use this when validating CORS, frontend wiring, or the actual upload pipeline.

Toggle between them by setting `MODE` below.

In [2]:
MODE = "in_process"  # or "http"
BASE = "http://localhost:8000/api/v1"

from pathlib import Path
FIXTURES = Path.cwd().parent / "tests" / "fixtures" / "broker_csvs"

if MODE == "in_process":
    from fastapi.testclient import TestClient
    from app.main import app
    client = TestClient(app)
    PREFIX = "/api/v1"
else:
    import httpx
    client = httpx.Client(base_url=BASE, timeout=30.0)
    PREFIX = ""

def get(path, **kw):
    r = client.get(f"{PREFIX}{path}", **kw)
    return r.status_code, r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text

def post(path, **kw):
    r = client.post(f"{PREFIX}{path}", **kw)
    return r.status_code, r.json() if r.headers.get("content-type", "").startswith("application/json") else r.text

print("Mode:", MODE)

ValueError: invalid literal for int() with base 10: '${POSTGRES_PORT:-5432}'

## 1. List configured sources

Six sources ship out of the box: 5 CSV (Zerodha, Coin, Groww, Dezerv, Wint) and 1 API (Angel One).

In [ ]:
import json
_, body = get("/portfolio/sources")
print(json.dumps(body, indent=2))

## 2. Upload sample CSVs

Pull each fixture from `tests/fixtures/broker_csvs/` and POST it. Real exports go in the same shape — see `backend/docs/BROKERS.md` for which menu item produces which file at each broker.

In [ ]:
FIXTURES_MAP = {
    "zerodha":      "zerodha_holdings.csv",
    "zerodha-coin": "zerodha_coin_holdings.csv",
    "groww":        "groww_stocks.csv",
    "dezerv":       "dezerv_holdings.csv",
    "wint-wealth":  "wint_wealth_holdings.csv",
}
for slug, fname in FIXTURES_MAP.items():
    path = FIXTURES / fname
    with path.open("rb") as f:
        status, body = post(f"/portfolio/sources/{slug}/upload", files={"file": (fname, f, "text/csv")})
    print(f"  {slug:14} → {status} ({body.get('holdings_count', '?')} holdings)" if status == 200 else f"  {slug:14} → {status} {body}")

## 3. Aggregate view

After upload, all sources are merged into one portfolio. Allocation is computed by `asset_class`.

In [ ]:
_, body = get("/portfolio/holdings")
print("Totals:", body["totals"])
print("\nAllocation:")
for s in body["allocation"]:
    print(f"  {s['asset_class']:12} ₹{s['value']:>14,.0f}  ({s['pct']:>5.1f}%)")
print("\nFirst 3 holdings:")
for h in body["holdings"][:3]:
    print("  ", h["source"], h["symbol"], h["current_value"])

## 4. Treemap layout

Pre-computed squarified layout — frontend just absolute-positions cells with the supplied `left/top/width/height` percentages.

In [ ]:
_, body = get("/portfolio/treemap")
for c in body["cells"][:8]:
    print(f"  {c['symbol']:14} {c['pct']:>5.1f}% @ ({c['left_pct']:>5.1f}, {c['top_pct']:>5.1f}) {c['width_pct']:>5.1f}x{c['height_pct']:>5.1f}")

## 5. Rebalance suggestions

Drift = actual - target. Default targets are 60/15/15/5/3/2 (equity/MF/bond/gold/crypto/cash). Suggestions fire above ±5%.

In [ ]:
_, body = get("/portfolio/rebalance")
print("Drift:")
for d in body["drift"]:
    print(f"  {d['asset_class']:12} target {d['target_pct']:>5.1f}% · actual {d['actual_pct']:>5.1f}% · drift {d['drift_pct']:>+5.1f}%")
print("\nSuggestions:")
for s in body["suggestions"]:
    print("  -", s["action"])

## 6. Angel One sync (live HTTP only)

Requires the four env vars set in `backend/.env`:

```
ANGEL_ONE_API_KEY=...
ANGEL_ONE_CLIENT_CODE=...
ANGEL_ONE_PASSWORD=...
ANGEL_ONE_TOTP_SECRET=... # base32 secret
```

Without them the sync raises a clear `RuntimeError`. See `backend/docs/BROKERS.md`.

In [ ]:
status, body = post("/portfolio/sources/angel-one/sync")
print(status, body)

## 7. Reset state

Clears the in-memory cache for one source so you can re-upload a different export.

In [ ]:
for slug in FIXTURES_MAP:
    post(f"/portfolio/sources/{slug}/reset")
_, body = get("/portfolio/sources")
for s in body["sources"]:
    print(f"  {s['slug']:14} {s['status']}  ({s['holdings_count']} holdings)")